In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
khanfashee_nih_chest_x_ray_14_224x224_resized_path = kagglehub.dataset_download('khanfashee/nih-chest-x-ray-14-224x224-resized')

print('Data source import complete.')


# Download Dependencies and Setup

In [ ]:
!pip install wandb -qU

# Data Preparation and Splitting

## Import Dependencies

In [ ]:
import os
import wandb
import random
import warnings

import cv2
import numpy as np
import pandas as pd
from PIL import Image
import albumentations as A
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt


import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import LinearLR, SequentialLR, CosineAnnealingLR, ReduceLROnPlateau


import timm
from timm.data import create_transform

from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

## Setup for Reproducability

In [ ]:
def set_seed(seed=42):
    """
    Sets all seeds for reproducibility.
    """
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # For multi-GPU setups

    # Critical for CuDNN reproducibility
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print(f"Seeds set to {seed} for reproducibility.")

set_seed(42)

## Define Constants and Paths

In [ ]:
# Define dataset paths for the Kaggle environment
DATA_DIR = '/kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized'
CSV_PATH = os.path.join(DATA_DIR, 'Data_Entry_2017.csv')
IMAGE_DIR = os.path.join(DATA_DIR, 'images-224/images-224/')

# 14 Pathology classes evaluated in ChestX-ray14
DISEASES = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion',
    'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule',
    'Pleural_Thickening', 'Pneumonia', 'Pneumothorax'
]

## Load the Dataset

In [ ]:
df = pd.read_csv(CSV_PATH)

# Add full image path for the PyTorch dataloader later
df['Image_Path'] = df['Image Index'].apply(lambda x: os.path.join(IMAGE_DIR, x))

print(f"Total records loaded: {len(df)}")
df.head()

## Preprocess and Binarize Labels

In [ ]:
# Split the pipe-separated strings into lists, handling 'No Finding' as an empty list
df['Labels'] = df['Finding Labels'].apply(lambda x: x.split('|') if x != 'No Finding' else [])

# Initialize and fit the binarizer for the 14 specific diseases
mlb = MultiLabelBinarizer(classes=DISEASES)
encoded_labels = mlb.fit_transform(df['Labels'])

# Append encoded labels as new columns to the dataframe
for i, disease in enumerate(DISEASES):
    df[disease] = encoded_labels[:, i]

df

## Patient-Wise Splitting

In [ ]:
# Initialize GroupShuffleSplit for the first split (Hold out 20% for Testing)
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# 1. Split into (Train+Val) and Test
train_val_idx, test_idx = next(gss_test.split(df, groups=df['Patient ID']))
train_val_df = df.iloc[train_val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

# Initialize a second GroupShuffleSplit for the Train/Val split
# Following the paper: 80% of the remaining data forms the training split
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# 2. Split (Train+Val) into Train and Validation
train_idx, val_idx = next(gss_val.split(train_val_df, groups=train_val_df['Patient ID']))
train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

print(f"Total distinct patients: {df['Patient ID'].nunique()}")
print(f"Training images: {len(train_df)} ({(len(train_df)/len(df))*100:.1f}%)")
print(f"Validation images: {len(val_df)} ({(len(val_df)/len(df))*100:.1f}%)")
print(f"Testing images: {len(test_df)} ({(len(test_df)/len(df))*100:.1f}%)")

# Verify absolutely no patient overlap across any of the three sets
train_patients = set(train_df['Patient ID'])
val_patients = set(val_df['Patient ID'])
test_patients = set(test_df['Patient ID'])

assert len(train_patients.intersection(val_patients)) == 0, "Leakage: Train/Val overlap"
assert len(train_patients.intersection(test_patients)) == 0, "Leakage: Train/Test overlap"
assert len(val_patients.intersection(test_patients)) == 0, "Leakage: Val/Test overlap"

print("Verification passed: No patient overlap across Train, Validation, and Test sets.")

# Dataset and DataLoaders

# Calculate Mean and Standard Deviation
This may take some time

In [ ]:
# def compute_mean_std(df, image_col="Image_Path", sample_size=5000, img_size=224):
#     """
#     Computes per-channel mean and std over a random sample of images.

#     Args:
#         df         : DataFrame containing image paths
#         image_col  : Column name with full image paths
#         sample_size: Number of images to sample (full dataset is slow)
#         img_size   : Images are resized to this before computing stats

#     Returns:
#         mean (np.ndarray): Per-channel mean, shape (3,)
#         std  (np.ndarray): Per-channel std,  shape (3,)
#     """
#     sample_df = df.sample(min(sample_size, len(df)), random_state=42)

#     pixel_sum    = np.zeros(3, dtype=np.float64)
#     pixel_sq_sum = np.zeros(3, dtype=np.float64)
#     pixel_count  = 0

#     for path in sample_df[image_col]:
#         img = np.array(
#             Image.open(path).convert("RGB").resize((img_size, img_size))
#         ) / 255.0                           # (H, W, 3) in [0, 1]

#         pixel_sum    += img.sum(axis=(0, 1))
#         pixel_sq_sum += (img ** 2).sum(axis=(0, 1))
#         pixel_count  += img_size * img_size

#     mean = pixel_sum / pixel_count
#     std  = np.sqrt(pixel_sq_sum / pixel_count - mean ** 2)

#     print(f"Dataset mean : {mean.round(4).tolist()}")
#     print(f"Dataset std  : {std.round(4).tolist()}")

#     return mean.astype(np.float32), std.astype(np.float32)


# DATASET_MEAN, DATASET_STD = compute_mean_std(train_df, sample_size=30_000)

In [ ]:
IMG_SIZE = 224
DATASET_MEAN = np.array([0.4982, 0.4982, 0.4982])
DATASET_STD = np.array([0.2484, 0.2484, 0.2484])

print("Building train, validation, and test transforms...")

# 1. Base transform (always applied): Resize + Normalize
base_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE, interpolation=cv2.INTER_CUBIC),
    A.Normalize(mean=DATASET_MEAN, std=DATASET_STD),
    ToTensorV2(),
])

# 3. Combined transform for training (both base + augmentation)
train_transform = A.Compose([
    # --- Geometric ---
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.15,        # was 0.05
        scale_limit=0.20,        # was 0.10
        rotate_limit=15,         # was 5
        border_mode=cv2.BORDER_CONSTANT, fill=0, p=0.75,  # was 0.5
    ),
    A.ElasticTransform(
        alpha=1.0, sigma=20,     # subtle tissue-like deformation
        p=0.3
    ),
    A.GridDistortion(
        num_steps=5, distort_limit=0.15,
        border_mode=cv2.BORDER_CONSTANT, p=0.2
    ),

    # --- Intensity / Contrast ---
    A.RandomBrightnessContrast(
        brightness_limit=0.35,   # was 0.2
        contrast_limit=0.35,     # was 0.2
        p=0.8                    # was 0.7
    ),
    A.RandomGamma(gamma_limit=(60, 140), p=0.4),   # was (80,120), 0.3
    A.CLAHE(clip_limit=4.0, tile_grid_size=(6, 6), p=0.5),  # was 4.0, 0.3
    A.GaussNoise(std_range=(0.01, 0.05), p=0.4),    # was (0.01, 0.05), 0.3

    # --- Always last ---
    A.Resize(IMG_SIZE, IMG_SIZE, interpolation=cv2.INTER_CUBIC),
    A.Normalize(mean=DATASET_MEAN, std=DATASET_STD),
    ToTensorV2(),
])

# Augmentation Visualizer for Training

In [ ]:
def visualize_augmentations(df, transform):
    """
    Picks a random image and shows one panel per augmentation,
    each applied in isolation, in a 2x5 grid.
    """
    SKIP = {"Resize", "Normalize", "ToTensorV2"}

    # Finalization transforms always applied at the end
    finalize = A.Compose([
        A.Resize(224, 224, interpolation=cv2.INTER_CUBIC),
        A.Normalize(mean=DATASET_MEAN.tolist(), std=DATASET_STD.tolist()),
        ToTensorV2(),
    ])

    def to_displayable(tensor):
        img_np = tensor.permute(1, 2, 0).numpy()
        return np.clip(img_np * DATASET_STD + DATASET_MEAN, 0, 1)

    # ── 1. Pick a random image ──────────────────────────────────────────────
    row = df.sample(1).iloc[0]
    img_path = row["Image_Path"]
    raw      = np.array(Image.open(img_path).convert("RGB"))

    active_diseases = [d for d in DISEASES if row[d] == 1]
    label_str = ", ".join(active_diseases) if active_diseases else "No Finding"

    # ── 2. Collect the meaningful transforms ───────────────────────────────
    aug_transforms = [
        t for t in transform.transforms
        if type(t).__name__ not in SKIP
    ]

    # ── 3. Build grid: original + one panel per augmentation ───────────────
    total  = 1 + len(aug_transforms)
    n_cols = 4
    n_rows = (total + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3.5))
    axes = axes.flatten()

    # ── 4. Original panel ───────────────────────────────────────────────────
    result = finalize(image=raw)
    axes[0].imshow(to_displayable(result["image"]))
    axes[0].set_title(f"ORIGINAL\n{label_str}", fontsize=7,
                      color="steelblue", fontweight="bold")
    axes[0].axis("off")

    # ── 5. One panel per augmentation, applied with p=1.0 ──────────────────
    for i, aug in enumerate(aug_transforms, start=1):
        name = type(aug).__name__

        # Force p=1.0 so the transform always fires
        isolated = A.Compose([
            aug.__class__(**{**aug.get_transform_init_args(), "p": 1.0}),
        ])

        augmented = isolated(image=raw)["image"]
        result    = finalize(image=augmented)

        axes[i].imshow(to_displayable(result["image"]))
        axes[i].set_title(name, fontsize=8, fontweight="bold")
        axes[i].axis("off")

    for j in range(total, len(axes)):
        axes[j].axis("off")

    fig.suptitle(
        f"Augmentation Preview  |  {os.path.basename(img_path)}",
        fontsize=11, fontweight="bold", y=1.01
    )
    plt.tight_layout()
    plt.show()

visualize_augmentations(train_df, train_transform)


## Create the Custom PyTorch Dataset Class

In [ ]:
class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        """
        Args:
            dataframe (pd.DataFrame): The train or validation dataframe.
            transform (callable, optional): Transform pipeline (augmentation + resize + normalize).
                                            Pass train_transform for training, base_transform for val/test.
        """
        self.dataframe = dataframe
        self.transform = transform
        self.image_paths = dataframe['Image_Path'].values
        self.labels = dataframe[DISEASES].values

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        image = np.array(Image.open(self.image_paths[idx]).convert('RGB'))

        if self.transform:
            image = self.transform(image=image)["image"]

        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return image, label

## Custom Sampler for Dynamic Augmentation

In [ ]:
def build_mixed_sampler(df, smoothed_aucs, diseases, mix_alpha=0.7):
    difficulty = np.clip(1.0 - smoothed_aucs, 0.05, 0.5)
    difficulty = np.sqrt(difficulty)

    # Matrix multiply: (N, C) x (C,) → (N,) in one shot
    label_matrix = df[diseases].values                        # (N, C)
    diff_weights = label_matrix @ difficulty                  # (N,)

    # Samples with no positive label get baseline weight 1.0
    diff_weights = np.where(diff_weights == 0, 1.0, diff_weights)

    final_weights = mix_alpha * diff_weights + (1 - mix_alpha)

    return WeightedRandomSampler(
        weights=torch.DoubleTensor(final_weights),
        num_samples=len(final_weights),
        replacement=True,
    )

## Instantiate Datasets and DataLoaders

In [ ]:
EMA_ALPHA = 0.3
MIX_ALPHA = 0.5


smoothed_aucs = np.full(len(DISEASES), 0.5, dtype=np.float64)

best_val_auroc = 0.0
model_save_path = "best_model.pth"

# Define batch size
BATCH_SIZE = 64  # Warning: Consider lowering to 32/64 if CUDA Out of Memory occurs


train_dataset = ChestXrayDataset(
    dataframe=train_df,
    transform=train_transform,  # Only augmentations
)

# Validation dataset (no augmentation, just base transforms)
val_dataset = ChestXrayDataset(
    dataframe=val_df,
    transform=base_transform,                     # No augmentations
)
test_dataset = ChestXrayDataset(
    dataframe=test_df,
    transform=base_transform,                     # No augmentations
)

sampler = build_mixed_sampler(train_df, smoothed_aucs, DISEASES, MIX_ALPHA)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=4,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,      # No need to shuffle test data
    num_workers=4,
    pin_memory=True
)

print(f"DataLoaders created. Batch size: {BATCH_SIZE}")
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Testing batches: {len(test_loader)}")

# Quick sanity check
sample_images, sample_labels = next(iter(train_loader))
print(f"Sample image batch shape: {sample_images.shape}")
print(f"Sample label batch shape: {sample_labels.shape}")

# Visualize TrainLoader Samples

In [ ]:
def visualize_train_samples(train_loader, class_names):
    images, labels = next(iter(train_loader))
    images = images.cpu()
    labels = labels.cpu()

    # 1. Define ImageNet stats as tensors reshaped for batch broadcasting (1, C, 1, 1)
    mean = torch.tensor(DATASET_MEAN).view(1, 3, 1, 1)
    std = torch.tensor([DATASET_STD]).view(1, 3, 1, 1)

    # 2. Correctly reverse the ImageNet normalization
    images = images * std + mean

    # 3. Clamp values to [0, 1] to ensure matplotlib doesn't complain about floating point rounding errors
    images = torch.clamp(images, 0, 1)

    fig, axes = plt.subplots(3, 4, figsize=(12, 9))
    axes = axes.flatten()

    for i in range(12):
        img = images[i]

        # Move channel dimension to the end for matplotlib (C, H, W) -> (H, W, C)
        img_np = img.permute(1, 2, 0).numpy()

        # Plot RGB
        axes[i].imshow(img_np)

        # Handle NIH multi-label parsing
        active = torch.where(labels[i] >= 0.5)[0]

        if active.numel() == 0:
            label_text = "No Finding"
        else:
            label_text = ", ".join([class_names[j] for j in active.tolist()])

        axes[i].set_title(label_text, fontsize=8)
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

# Run the updated visualizer
visualize_train_samples(train_loader, class_names=DISEASES)

# Model Architecture

## Main CheXNet Architecture

In [ ]:
class DenseCheX(nn.Module):
    def __init__(self, num_classes=14):
        super().__init__()
        # By passing num_classes=14, timm automatically drops the 1000-class
        # ImageNet head and replaces it with a single nn.Linear(1024, 14) layer.
        self.model = timm.create_model(
            'densenet121',
            pretrained=True,
            num_classes=num_classes
        )

    def forward(self, x):
        # Outputs raw logits for the 14 classes to be used with BCEWithLogitsLoss
        return self.model(x)

## Instantiate and Prepare for Dual GPUs
Creating the model object, checking for available GPUs, and wrapping the model in nn.DataParallel

In [ ]:
# Check hardware availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")
print(f"Number of GPUs available: {torch.cuda.device_count()}")

# Instantiate the model
model = DenseCheX(num_classes=len(DISEASES))

# Wrap model for multi-GPU training if more than 1 GPU is available
if torch.cuda.device_count() > 1:
    print("Dual GPUs detected! Wrapping model in nn.DataParallel...")
    model = nn.DataParallel(model)

# Move the model to the GPUs
model = model.to(device)

print("Model successfully instantiated and moved to device.")

## Test Model

In [ ]:
# 1. Grab a single batch of data
sample_images, sample_labels = next(iter(train_loader))

# 2. Move the data to the correct device (GPUs)
sample_images = sample_images.to(device)
sample_labels = sample_labels.to(device)

print(f"Input images shape: {sample_images.shape}")

# 3. Put the model in evaluation mode for the test
model.eval()

# 4. Perform a forward pass (no gradients needed)
with torch.no_grad():
    outputs = model(sample_images)

# 5. Verify the output
print(f"Model output shape: {outputs.shape}")
print(f"Expected output shape: torch.Size([{sample_images.shape[0]}, {len(DISEASES)}])")

# 6. Strict assertion to ensure shapes match
assert outputs.shape == (sample_images.shape[0], len(DISEASES)), "Error: Output shape mismatch!"
print("Sanity check passed successfully! The model correctly processed the batch.")

# Training & Validation Pipeline

## Define the Loss Function and Optimizer

In [ ]:
class WeightedFocalLoss(nn.Module):
    def __init__(self, pos_weight=None, gamma=2.0, reduction='mean'):
        super(WeightedFocalLoss, self).__init__()
        self.register_buffer('pos_weight', pos_weight)  # moves to device with .to(device)
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # --- 1. Weighted BCE (numerically stable, no explicit sigmoid) ---
        bce_loss = F.binary_cross_entropy_with_logits(
            inputs,
            targets,
            pos_weight=self.pos_weight,
            reduction='none'
        )

        # --- 2. Focal modulation ---
        # pt = probability assigned to the TRUE label (high pt = easy example)
        pt = torch.exp(-bce_loss)
        focal_loss = (1 - pt) ** self.gamma * bce_loss

        # --- 3. Reduction ---
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [ ]:
# Compute pos_weight from training label frequencies
label_counts = train_df[DISEASES].sum()                          # positive count per class
neg_counts   = len(train_df) - label_counts                      # negative count per class
pos_weight   = torch.tensor(
    (neg_counts / (label_counts + 1e-6)).values,
    dtype=torch.float32
).to(device)

criterion = torch.nn.BCEWithLogitsLoss()

EPOCHS = 25
LEARNING_RATE = 1e-4

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01,
)
print("Optimizer configured with weight decay.")

## Sequential Learning Rate

In [ ]:
EPOCHS = 25
WARMUP_EPOCHS = 3

warmup_scheduler = LinearLR(
    optimizer,
    start_factor=0.01,
    end_factor=1.0,
    total_iters=WARMUP_EPOCHS
)

cosine_scheduler = CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS - WARMUP_EPOCHS,   # Cosine cycle over the remaining epochs
    eta_min=1e-7,                    # Minimum LR at the end of the cycle
)

scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[WARMUP_EPOCHS]
)

## Define Comprehensive Clinical Metrics

In [ ]:
def calculate_comprehensive_metrics(y_true, y_pred_probs, threshold=0.5):
    # Convert probabilities to binary predictions (0 or 1) based on the threshold
    y_pred_binary = (y_pred_probs >= threshold).astype(int)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        # Calculate standard metrics (Macro-averaged across all 14 classes)
        # Recall is the same as Sensitivity in this context
        metrics = {
            'accuracy': accuracy_score(y_true, y_pred_binary), # Exact match accuracy (strict in multi-label)
            'f1_macro': f1_score(y_true, y_pred_binary, average='macro', zero_division=0),
            'precision_macro': precision_score(y_true, y_pred_binary, average='macro', zero_division=0),
            'sensitivity_macro': recall_score(y_true, y_pred_binary, average='macro', zero_division=0)
        }

        # Calculate Specificity (True Negative Rate) for each class, then macro-average
        specificities = []
        for i in range(y_true.shape[1]):
            # ravel() unwraps the confusion matrix into TN, FP, FN, TP
            tn, fp, fn, tp = confusion_matrix(y_true[:, i], y_pred_binary[:, i], labels=[0, 1]).ravel()

            # Specificity = TN / (TN + FP)
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
            specificities.append(specificity)

        metrics['specificity_macro'] = np.mean(specificities)

    return metrics

# The Training Loop & Checkpointing

## Define the Training Epoch Function

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, scaler):
    model.train()
    running_loss = 0.0
    progress_bar = tqdm(dataloader, desc="Training", leave=False)

    for images, labels in progress_bar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})

    return running_loss / len(dataloader.dataset)

## Define the Validation Epoch Function

In [ ]:
from sklearn.metrics import f1_score

def find_optimal_thresholds(labels, preds):
    """Find per-class threshold that maximizes F1."""
    thresholds = np.arange(0.1, 0.9, 0.05)
    optimal = []
    for c in range(labels.shape[1]):
        best_t, best_f1 = 0.5, 0.0
        for t in thresholds:
            f1 = f1_score(labels[:, c], (preds[:, c] >= t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        optimal.append(best_t)
    return np.array(optimal)


def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    progress_bar = tqdm(dataloader, desc="Validating", leave=False)

    with torch.no_grad():
        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)

            with torch.amp.autocast("cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            all_preds.append(torch.sigmoid(outputs).cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    all_preds  = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    # --- AUC (primary) ---
    per_class_aucs = roc_auc_score(all_labels, all_preds, average=None)
    epoch_auroc    = np.mean(per_class_aucs)

    # --- F1 with optimal per-class thresholds (diagnostic) ---
    optimal_thresholds = find_optimal_thresholds(all_labels, all_preds)
    binary_preds       = (all_preds >= optimal_thresholds).astype(int)
    per_class_f1       = f1_score(all_labels, binary_preds, average=None, zero_division=0)
    macro_f1           = np.mean(per_class_f1)

    return epoch_loss, epoch_auroc, per_class_aucs, macro_f1, per_class_f1, optimal_thresholds

## Execute the Main Loop and Save the Best Model

In [ ]:
# 2. Login programmatically
wandb.login(key="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")

# Silence the extra wandb logs so your console stays clean
os.environ["WANDB_SILENT"] = "true"

In [ ]:
scaler = torch.amp.GradScaler("cuda")

In [ ]:
run = wandb.init(
    project="nih-chestx-ray-classification",
    config={
        "epochs":             EPOCHS,
        "learning_rate":      LEARNING_RATE,
        "loss_function":      "Binary Crossentropy",
        "augmentation":       "Geometric+Intensity+Elastic",
        "model_arch":         "DenseCheX",
        "sampler_strategy":   "Dynamic Weighted Sampling",
        "sampler_ema_alpha":  EMA_ALPHA,
        "sampler_mix_alpha":  MIX_ALPHA,
    },
)
config = wandb.config

best_val_auroc  = 0.0
model_save_path = "best_model.pth"

print(f"Starting training for {config.epochs} epochs...")
print(f"Dataset size: {len(train_dataset)}")

for epoch in range(config.epochs):
    print(f"\nEpoch {epoch + 1}/{config.epochs}")

    # ------------------------------
    # TRAIN
    # ------------------------------
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler)

    # ------------------------------
    # VALIDATION
    # ------------------------------
    val_loss, val_auroc, per_class_aucs, macro_f1, per_class_f1, optimal_thresholds = \
        validate_one_epoch(model, val_loader, criterion, device)

    # ------------------------------
    # SCHEDULER STEP
    # ------------------------------
    scheduler.step()

    # ------------------------------
    # LOGGING
    # ------------------------------
    per_class_auc_log = {f"auc/{d}":  auc for d, auc  in zip(DISEASES, per_class_aucs)}
    per_class_f1_log  = {f"f1/{d}":   f1  for d, f1   in zip(DISEASES, per_class_f1)}
    threshold_log     = {f"thresh/{d}": t  for d, t    in zip(DISEASES, optimal_thresholds)}
    sampler_log       = {
        f"sampler_weight/{d}": MIX_ALPHA * (1.0 - auc) + (1 - MIX_ALPHA)
        for d, auc in zip(DISEASES, smoothed_aucs)
    }

    wandb.log({
        "epoch":          epoch + 1,
        "train/loss":     train_loss,
        "val/loss":       val_loss,
        "val/auroc":      val_auroc,       # primary metric
        "val/f1_macro":   macro_f1,        # diagnostic
        "lr":             optimizer.param_groups[0]["lr"],
        **per_class_auc_log,
        **per_class_f1_log,
        **threshold_log,                   # track threshold drift per class
        **sampler_log,
    })

    # ------------------------------
    # CHECKPOINT (AUC is still the primary metric)
    # ------------------------------
    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        wandb.run.summary["best_val_auroc"] = best_val_auroc
        wandb.run.summary["best_val_f1_macro"] = macro_f1  # log F1 at best AUC checkpoint

        state = (
            model.module.state_dict()
            if isinstance(model, torch.nn.DataParallel)
            else model.state_dict()
        )
        torch.save(state, model_save_path)
        wandb.save(model_save_path)
        print(f"✓ New best checkpoint saved (val AUROC: {best_val_auroc:.4f} | F1: {macro_f1:.4f})")

    # ------------------------------
    # UPDATE SAMPLER (after warmup)
    # ------------------------------
    if epoch >= WARMUP_EPOCHS:
        smoothed_aucs = EMA_ALPHA * per_class_aucs + (1 - EMA_ALPHA) * smoothed_aucs

        underperforming = [
            d for d, auc in zip(DISEASES, smoothed_aucs)
            if auc < smoothed_aucs.mean()
        ]
        print(
            f"Smoothed avg AUC: {smoothed_aucs.mean():.4f} | "
            f"Classes under pressure: {underperforming}"
        )

        sampler = build_mixed_sampler(train_df, smoothed_aucs, DISEASES, MIX_ALPHA)
        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=4,
            pin_memory=True,
        )

wandb.finish()

# Inference and Testing

## Load the best model

In [ ]:
print("Loading the best model for testing...")

# 1. Instantiate the base model (using Swin-Base as we configured)
test_model = DenseCheX(num_classes=len(DISEASES))

# 2. Load the saved weights (use the same relative path as training)
model_save_path = "best_model.pth"

if os.path.exists(model_save_path):
    # We saved model.module.state_dict() during training, so we load directly into the base model
    test_model.load_state_dict(torch.load(model_save_path))
    print("Successfully loaded optimal weights from checkpoint.")
else:
    print(f"Error: Could not find checkpoint at {model_save_path}")

# 3. Move to device and wrap in DataParallel for the Dual T4s
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.device_count() > 1:
    test_model = nn.DataParallel(test_model)

test_model = test_model.to(device)
test_model.eval() # CRITICAL: Set to evaluation mode to disable dropout/batchnorm updates

## Perform inference on test

In [ ]:
print("Running inference on the unseen Test Set...")

all_test_preds = []
all_test_labels = []

# Create a progress bar
test_progress = tqdm(test_loader, desc="Testing", leave=False)

with torch.no_grad():
    for images, labels in test_progress:
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = test_model(images)

        # Convert to probabilities
        probabilities = torch.sigmoid(outputs)

        # Move to CPU and store
        all_test_preds.append(probabilities.cpu().numpy())
        all_test_labels.append(labels.cpu().numpy())

# Stack the batches into full numpy arrays
test_preds_np = np.vstack(all_test_preds)
test_labels_np = np.vstack(all_test_labels)

print(f"Inference complete. Total test images processed: {test_labels_np.shape[0]}")

## Calculate performance for each class

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_curve, f1_score
import numpy as np
import pandas as pd
import warnings

print("Calculating detailed F1 breakdown...\n")

results = []

with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    for i, disease in enumerate(DISEASES):
        true_labels = test_labels_np[:, i]
        pred_probs = test_preds_np[:, i]

        # 1. Calculate AUROC
        try:
            auc = roc_auc_score(true_labels, pred_probs)
        except ValueError:
            auc = np.nan

        # 3. Apply the threshold and calculate the FULL breakdown
        pred_binary = (pred_probs >= 0.5).astype(int)

        # Calculate the Negative F1 and Macro F1 explicitly
        # Note: zero_division=0 prevents warnings if a class is entirely missed
        macro_f1 = f1_score(true_labels, pred_binary, average='macro', zero_division=0)

        acc = accuracy_score(true_labels, pred_binary)

        results.append({
            'Pathology': disease,
            'F1': macro_f1,
            'AUROC': auc
        })

# Create DataFrame
results_df = pd.DataFrame(results)

# Append Means
mean_row = pd.DataFrame([{
    'Pathology': 'MEAN',
    'F1': results_df['F1'].mean(),
    'AUROC': results_df['AUROC'].mean()
}])
results_df = pd.concat([results_df, mean_row], ignore_index=True)

# Format for clean output
for col in ['F1', 'AUROC']:
    results_df[col] = results_df[col].map('{:.3f}'.format)

display(results_df)